<a href="https://colab.research.google.com/github/saeid-uot/3253-Machine-Learning/blob/main/Week_08%20-%20Dimensionality%20Reduction/1-%20Solution%20PCA%20MNIST%20Student%20Practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Objective:

### Use MNIST dataset and apply PCA to find out the impact on the model training time and also model performance
### The work is taken from https://github.com/mGalarnyk/Python_Tutorials/blob/master/Sklearn/PCA/PCA_to_Speed-up_Machine_Learning_Algorithms.ipynb

In [1]:
# Setup
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn import metrics
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_openml


#Download and Load the Data


In [2]:
from sklearn.datasets import fetch_openml

mnist = fetch_openml('mnist_784', version=1, cache=True)
mnist.target = mnist.target.astype(np.int8) # fetch_openml() returns targets as strings

X, y = mnist["data"], mnist["target"]

# Split data into train/test

In [3]:
# Write a code to split your dataset into 80/20 dataset
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size= 0.2)

# View Data Dimension

In [4]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape


((56000, 784), (14000, 784), (56000,), (14000,))

#Standardizing the Data¶

Since PCA yields a feature subspace that maximizes the variance along the axes, it makes sense to standardize the data, especially, if it was measured on different scales.

Standardization of a dataset is a common requirement for many machine learning estimators: they might behave badly if the individual feature do not more or less look like standard normally distributed data

Notebook going over the importance of feature Scaling: http://scikit-learn.org/stable/auto_examples/preprocessing/plot_scaling_importance.html#sphx-glr-auto-examples-preprocessing-plot-scaling-importance-py


In [5]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

# Fit on training set only.
scaler.fit(X_train)

# Apply transform to both the training set and the test set.
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

#X_train.shape, X_test.shape, y_train.shape, y_test.shape


# Lets fit a simple model

In [6]:
from sklearn.linear_model import LogisticRegression
logisticRegr = LogisticRegression(max_iter=1000)

import datetime
start= datetime.datetime.now()
logisticRegr.fit(X_train, y_train)
end= datetime.datetime.now()

#time taken to train the model
print(end-start)

0:01:15.131702


#Measure the accuracy of the model before PCA

In [7]:
from sklearn.metrics import accuracy_score

acc_train = accuracy_score(y_train, logisticRegr.predict(X_train))
acc_test = accuracy_score(y_test, logisticRegr.predict(X_test))

print("Performance on train:" , acc_train ,"\n", "Performance on test:", acc_test)

Performance on train: 0.9440714285714286 
 Performance on test: 0.9140714285714285


In [8]:
# In case you want to see how the scaled number would look like, you can uncomment below lines
#from scipy.stats import describe
#describe(X_train)[1]

#Now, lets implement PCA

In [9]:
from sklearn.decomposition import PCA
# specify how much of variation you would like PCA to capture (between 0-1)
# pca = PCA('mle')
pca = PCA(0.5)

pca.fit(X_train)


PCA(n_components=0.5)

# Look at components

In [10]:
pca.n_components_


np.int64(38)

#Apply the mapping (transform) to both the training set and the test set.



In [11]:
X_train = pca.transform(X_train)
X_test = pca.transform(X_test)

#Build a linear model and measure model fitting period.

In [12]:
from sklearn.linear_model import LogisticRegression
logisticRegr = LogisticRegression(max_iter=1000)

import datetime
start= datetime.datetime.now()
logisticRegr.fit(X_train, y_train)
end= datetime.datetime.now()

#measure training time

In [13]:

print(end-start)
#logisticRegr.predict(X_train[0].reshape(1,-1))


0:00:10.534430


#Measuring Model Performance

In [14]:
score = logisticRegr.score(X_test, y_test)
print(score)

0.8972857142857142


#Now lets build a loop to go through various value of PCA n_component, measure training time and accuracy

In [15]:
import pandas as pd

pca_values = [0.99, 0.98, 0.95, 0.90, 0.80, 0.6]
results = []

for pca_val in pca_values:
    pca = PCA(pca_val)
    pca.fit(X_train)
    X_train_pca = pca.transform(X_train)
    X_test_pca = pca.transform(X_test)

    logisticRegr = LogisticRegression(max_iter=1000) # Increased max_iter

    start_time = datetime.datetime.now()
    logisticRegr.fit(X_train_pca, y_train)
    end_time = datetime.datetime.now()

    training_time = end_time - start_time

    train_accuracy = accuracy_score(y_train, logisticRegr.predict(X_train_pca))
    test_accuracy = accuracy_score(y_test, logisticRegr.predict(X_test_pca))

    results.append([pca_val, round(training_time.total_seconds(), 3), train_accuracy, test_accuracy])


df = pd.DataFrame(results, columns=['PCA Value', 'Training Time', 'Train Accuracy', 'Test Accuracy'])
df


,PCA Value,Training Time,Train Accuracy,Test Accuracy
0,0.99,10.921,0.901089,0.897286
1,0.98,11.585,0.898768,0.895071
2,0.95,11.462,0.894946,0.891357
3,0.90,8.813,0.891732,0.888429
4,0.80,21.875,0.878214,0.872143
5,0.60,10.585,0.818786,0.815357


In [ ]:
# repeat the above code on a larger training set bigger than mnist to show the impact of PCA.

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn import metrics
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.linear_model import LogisticRegression
import datetime
from sklearn.metrics import accuracy_score

# Use a larger dataset than MNIST
# Example: Fashion-MNIST
fashion_mnist = fetch_openml('Fashion-MNIST', version=1, cache=True)
fashion_mnist.target = fashion_mnist.target.astype(np.int8)

X, y = fashion_mnist["data"], fashion_mnist["target"]

# Split data into train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42) # Added random_state for reproducibility

# Standardizing the Data
scaler = StandardScaler()
scaler.fit(X_train)
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

# Train a Logistic Regression model without PCA
logisticRegr = LogisticRegression(max_iter=1000) # Increased max_iter for convergence

start = datetime.datetime.now()
logisticRegr.fit(X_train, y_train)
end = datetime.datetime.now()

print("Training time without PCA:", end - start)

acc_train = accuracy_score(y_train, logisticRegr.predict(X_train))
acc_test = accuracy_score(y_test, logisticRegr.predict(X_test))

print("Performance without PCA:\nTrain accuracy:", acc_train, "\nTest accuracy:", acc_test)


# Apply PCA
pca = PCA(0.95) # Example: retain 95% of variance
pca.fit(X_train)
X_train_pca = pca.transform(X_train)
X_test_pca = pca.transform(X_test)

# Train a Logistic Regression model with PCA
logisticRegr_pca = LogisticRegression(max_iter=1000)

start_pca = datetime.datetime.now()
logisticRegr_pca.fit(X_train_pca, y_train)
end_pca = datetime.datetime.now()

print("\nTraining time with PCA:", end_pca - start_pca)

acc_train_pca = accuracy_score(y_train, logisticRegr_pca.predict(X_train_pca))
acc_test_pca = accuracy_score(y_test, logisticRegr_pca.predict(X_test_pca))

print("Performance with PCA:\nTrain accuracy:", acc_train_pca, "\nTest accuracy:", acc_test_pca)


Training time without PCA: 0:06:18.732587
Performance without PCA:
Train accuracy: 0.8857857142857143 
Test accuracy: 0.8445714285714285


#Number of Components, Variance, Time Table


In [ ]:
# In the past with less computing resources of Google Colab, I had below results. These are for illustraiton only.
pd.DataFrame(data = [[1.00, 784, 48.94, .9158],
                     [.99, 541, 34.69, .9169],
                     [.95, 330, 13.89, .92],
                     [.90, 236, 10.56, .9168],
                     [.85, 184, 8.85, .9156]],
             columns = ['Variance Retained',
                      'Number of Components',
                      'Time (seconds)',
                      'Accuracy'])

,Variance Retained,Number of Components,Time (seconds),Accuracy
0,1.00,784,48.94,0.9158
1,0.99,541,34.69,0.9169
2,0.95,330,13.89,0.9200
3,0.90,236,10.56,0.9168
4,0.85,184,8.85,0.9156


In [ ]:
#My own results

# pca(0.99)= n_componnets = mle (81), acc= 0.9167857142857143
# pca(0.85)= n_componnets = 150, acc= 0.916
# pca(0.7)= n_componnets = 53, acc= 0.906

